Chat con GPT y mongo

In [ ]:
pip install openai pymongo python-dotenv

OPENAI_API_KEY=sk-xxxxxx...
MONGO_URI=mongodb://localhost:27017


In [ ]:
import os
from datetime import datetime
from pymongo import MongoClient
import openai
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env
# Asegúrate de tener un archivo .env con OPENAI_API_KEY y MONGO_URI definidos
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MONGO_URI = os.getenv("MONGO_URI")

# Conexión a MongoDB
client = MongoClient(MONGO_URI)
db = client["chatdb"]
coleccion = db["chats"]

def generar_respuesta(prompt, modelo="gpt-3.5-turbo"):
    """
    Genera una respuesta con el modelo GPT usando OpenAI API.
    
    Parámetros:
        prompt (str): Texto ingresado por el usuario.
        modelo (str): Modelo de OpenAI a utilizar (default: gpt-3.5-turbo).
        
    Retorna:
        str: Texto de la respuesta generada.
        str: Nombre del modelo utilizado.
        str: ID de la respuesta de OpenAI.
    """
    try:
        respuesta = openai.ChatCompletion.create(
            model=modelo,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        contenido = respuesta['choices'][0]['message']['content']
        return contenido.strip(), modelo, respuesta['id']
    except Exception as e:
        print("❌ Error al generar respuesta:", e)
        return None, modelo, None

def guardar_en_mongodb(prompt, respuesta, modelo, respuesta_id):
    """
    Guarda en MongoDB el prompt, la respuesta y metadatos.

    Parámetros:
        prompt (str): Pregunta o entrada del usuario.
        respuesta (str): Texto generado por el modelo.
        modelo (str): Modelo de OpenAI utilizado.
        respuesta_id (str): ID de la respuesta en OpenAI.
    """
    documento = {
        "prompt": prompt,
        "respuesta": respuesta,
        "modelo": modelo,
        "respuesta_id": respuesta_id,
        "timestamp": datetime.now()
    }
    resultado = coleccion.insert_one(documento)
    print(f"✅ Documento insertado con ID: {resultado.inserted_id}")

def main():
    """
    Función principal que ejecuta el ciclo de chat.
    """
    print("🤖 Chat IA conectado a MongoDB. Escribe 'salir' para terminar.\n")
    while True:
        prompt = input("Tú: ")
        if prompt.lower() in ['salir', 'exit', 'quit']:
            print("👋 Terminando sesión.")
            break

        respuesta, modelo, respuesta_id = generar_respuesta(prompt)
        if respuesta:
            print(f"IA ({modelo}): {respuesta}\n")
            guardar_en_mongodb(prompt, respuesta, modelo, respuesta_id)
        else:
            print("⚠️ No se pudo generar respuesta.\n")

if __name__ == "__main__":
    main()
